<a href="https://colab.research.google.com/github/ks-orintis/machine-learning/blob/main/%D0%9B%D0%A09_%D0%9B%D1%83%D1%87%D0%B0%D0%BD%D1%96%D0%BD%D0%BE%D0%B2%D0%B0_%D0%9A%D1%81%D0%B5%D0%BD%D1%96%D1%8F_%D0%A4%D0%86%D0%A2_4_8_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Лучанінова Ксенія

**Група:** ФІТ 4-8  

**Дата:** 19.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [25]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [26]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from IPython.display import display, Markdown
from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV, KFold
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("pandas:", pd.__version__)
print("Surprise:", surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [27]:
data = Dataset.load_builtin("ml-100k", prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=["user_id", "item_id", "rating", "timestamp"]
)

print("Розмір таблиці:", ratings_df.shape)
display(ratings_df.head())

Розмір таблиці: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [28]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

print("Кількість користувачів:", ratings_df["user_id"].nunique())
print("Кількість фільмів:", ratings_df["item_id"].nunique())

print("Мінімальна оцінка:", ratings_df["rating"].min())
print("Максимальна оцінка:", ratings_df["rating"].max())
print("Середня оцінка:", round(ratings_df["rating"].mean(), 4))

print("\nКількість пропущених значень:")
display(ratings_df.isna().sum())

rating_distribution = (
    ratings_df["rating"]
    .value_counts()
    .sort_index()
    .rename_axis("Оцінка")
    .to_frame("Кількість")
)

rating_distribution["Частка, %"] = (
    rating_distribution["Кількість"] / len(ratings_df) * 100
).round(2)

print("\nРозподіл оцінок:")
display(rating_distribution)

Кількість користувачів: 943
Кількість фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.5299

Кількість пропущених значень:


,0
user_id,0
item_id,0
rating,0
timestamp,0



Розподіл оцінок:


,Кількість,"Частка, %"
Оцінка,,
1.0,6110,6.11
2.0,11370,11.37
3.0,27145,27.15
4.0,34174,34.17
5.0,21201,21.20


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

Набір даних містить 100 000 оцінок від 943 користувачів для 1682 фільмів. Оцінки змінюються від 1 до 5, середнє значення становить 3,5299, а найчастіше зустрічається оцінка 4 — 34,17% усіх записів. Пропущених значень у таблиці немає.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [29]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print("Навчальна вибірка:", train_df.shape)
print("Тестова вибірка:", test_df.shape)

Навчальна вибірка: (80000, 4)
Тестова вибірка: (20000, 4)


In [30]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[["user_id", "item_id", "rating"]],
    reader
)

testset = list(
    test_df[["user_id", "item_id", "rating"]]
    .itertuples(index=False, name=None)
)

print("Шкала оцінок:", reader.rating_scale)
print("Кількість навчальних оцінок:", len(train_data.raw_ratings))
print("Кількість тестових оцінок:", len(testset))
print("Приклад тестового запису:", testset[0])

Шкала оцінок: (1, 5)
Кількість навчальних оцінок: 80000
Кількість тестових оцінок: 20000
Приклад тестового запису: ('877', '381', 4.0)


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [31]:
param_grid_svd = {
    "n_factors": [50, 100],
    "lr_all": [0.005, 0.010],
    "reg_all": [0.02, 0.05],
    "random_state": [42]
}

param_grid_knn = {
    "k": [20, 40],
    "min_k": [1, 5],
    "sim_options": {
        "name": ["cosine", "pearson"],
        "user_based": [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

Для SVD обрано 50 і 100 прихованих факторів, щоб порівняти моделі різної складності. Значення швидкості навчання 0,005 і 0,010 та регуляризації 0,02 і 0,05 дають змогу дослідити вплив інтенсивності оновлення параметрів і сили їх обмеження на точність прогнозування. Для KNNBasic перевірено 20 і 40 сусідів, мінімальну кількість сусідів 1 і 5, косинусну схожість та кореляцію Пірсона, а також підходи на основі схожості користувачів і фільмів. Загалом перевірено 8 комбінацій параметрів SVD і 16 комбінацій KNNBasic з використанням трьох розбиттів перехресної перевірки.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [32]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

cv = KFold(
    n_splits=3,
    random_state=42,
    shuffle=True
)

grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=["rmse", "mae"],
    cv=cv,
    n_jobs=-1
)

grid_search_svd.fit(train_data)

print("Підбір параметрів SVD завершено")

Підбір параметрів SVD завершено


In [33]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=["rmse", "mae"],
    cv=cv,
    n_jobs=-1
)

grid_search_knn.fit(train_data)

print("Підбір параметрів KNNBasic завершено")

Підбір параметрів KNNBasic завершено


In [34]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

for model_name, search in [
    ("SVD", grid_search_svd),
    ("KNNBasic", grid_search_knn)
]:
    print(model_name)
    print(f"Найкращий CV RMSE: {search.best_score['rmse']:.4f}")
    print("Параметри за RMSE:", search.best_params["rmse"])
    print(f"Найкращий CV MAE: {search.best_score['mae']:.4f}")
    print("Параметри за MAE:", search.best_params["mae"])
    print()

SVD
Найкращий CV RMSE: 0.9425
Параметри за RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05, 'random_state': 42}
Найкращий CV MAE: 0.7441
Параметри за MAE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05, 'random_state': 42}

KNNBasic
Найкращий CV RMSE: 1.0272
Параметри за RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}
Найкращий CV MAE: 0.8143
Параметри за MAE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}



In [35]:
svd_best_index = grid_search_svd.best_index["rmse"]
knn_best_index = grid_search_knn.best_index["rmse"]

svd_cv_rmse = float(grid_search_svd.best_score["rmse"])
knn_cv_rmse = float(grid_search_knn.best_score["rmse"])

svd_cv_mae = float(
    grid_search_svd.cv_results["mean_test_mae"][svd_best_index]
)

knn_cv_mae = float(
    grid_search_knn.cv_results["mean_test_mae"][knn_best_index]
)

cv_results = pd.DataFrame({
    "Модель": ["SVD", "KNNBasic"],
    "CV RMSE": [svd_cv_rmse, knn_cv_rmse],
    "CV MAE": [svd_cv_mae, knn_cv_mae],
    "Параметри за RMSE": [
        str(grid_search_svd.best_params["rmse"]),
        str(grid_search_knn.best_params["rmse"])
    ]
})

display(cv_results)

selected_model_name = (
    "SVD" if svd_cv_rmse <= knn_cv_rmse else "KNNBasic"
)

,Модель,CV RMSE,CV MAE,Параметри за RMSE
0,SVD,0.942506,0.744138,"{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0..."
1,KNNBasic,1.027193,0.814330,"{'k': 40, 'min_k': 1, 'sim_options': {'name': ..."


За результатами перехресної перевірки SVD отримала RMSE = 0,9425 і MAE = 0,7441, тоді як KNNBasic — RMSE = 1,0272 і MAE = 0,8143. Найкращими параметрами SVD стали 50 прихованих факторів, швидкість навчання 0,01 і регуляризація 0,05. Для KNNBasic обрано 40 сусідів, мінімальну кількість сусідів 1 та косинусну схожість між користувачами. SVD має менші значення обох помилок, тому її обрано для формування рекомендацій за найменшим RMSE на перехресній перевірці.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [36]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

best_svd_params = grid_search_svd.best_params["rmse"].copy()
best_knn_params = grid_search_knn.best_params["rmse"].copy()

trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params)
best_knn = KNNBasic(**best_knn_params, verbose=False)

best_svd.fit(trainset)
best_knn.fit(trainset)

print("Обидві моделі навчено")
print("Кількість оцінок:", trainset.n_ratings)
print("Кількість користувачів:", trainset.n_users)
print("Кількість фільмів:", trainset.n_items)

Обидві моделі навчено
Кількість оцінок: 80000
Кількість користувачів: 943
Кількість фільмів: 1653


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [37]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print(f"SVD: RMSE = {svd_rmse:.4f}, MAE = {svd_mae:.4f}")
print(f"KNNBasic: RMSE = {knn_rmse:.4f}, MAE = {knn_mae:.4f}")

SVD: RMSE = 0.9192, MAE = 0.7234
KNNBasic: RMSE = 1.0120, MAE = 0.7984


In [38]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

results = pd.DataFrame({
    "Model": ["SVD", "KNNBasic"],
    "RMSE": [svd_rmse, knn_rmse],
    "MAE": [svd_mae, knn_mae]
})

results = results.sort_values("RMSE").reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


In [39]:
comparison = pd.DataFrame({
    "Модель": ["SVD", "KNNBasic"],
    "CV RMSE": [svd_cv_rmse, knn_cv_rmse],
    "Test RMSE": [svd_rmse, knn_rmse],
    "CV MAE": [svd_cv_mae, knn_cv_mae],
    "Test MAE": [svd_mae, knn_mae]
})

comparison["Test RMSE − CV RMSE"] = (
    comparison["Test RMSE"] - comparison["CV RMSE"]
)

comparison["Test MAE − CV MAE"] = (
    comparison["Test MAE"] - comparison["CV MAE"]
)

display(comparison)

,Модель,CV RMSE,Test RMSE,CV MAE,Test MAE,Test RMSE − CV RMSE,Test MAE − CV MAE
0,SVD,0.942506,0.919175,0.744138,0.723355,-0.023332,-0.020783
1,KNNBasic,1.027193,1.011953,0.814330,0.798421,-0.015241,-0.015909


На незалежній тестовій вибірці SVD отримала RMSE = 0,9192 і MAE = 0,7234, а KNNBasic — RMSE = 1,0120 і MAE = 0,7984. Для обох моделей тестові помилки дещо нижчі, ніж на перехресній перевірці: RMSE зменшився на 0,0233 для SVD і на 0,0152 для KNNBasic. Відмінності можуть бути пов’язані зі складом вибірок та навчанням остаточних моделей на всіх 80 000 навчальних оцінках. SVD зберегла перевагу за обома показниками, а тестові результати використано лише для остаточного оцінювання без повторного підбору параметрів.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [41]:
# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

user_id = "196"

if svd_cv_rmse <= knn_cv_rmse:
    selected_model = best_svd
    selected_model_name = "SVD"
else:
    selected_model = best_knn
    selected_model_name = "KNNBasic"

top_10 = pd.DataFrame(columns=["item_id", "predicted_rating"])

try:
    inner_user_id = trainset.to_inner_uid(user_id)
except ValueError:
    print(f"Користувача {user_id} немає в навчальній вибірці")
else:
    rated_items = {
        inner_item_id
        for inner_item_id, rating in trainset.ur[inner_user_id]
    }

    recommendations = []

    for inner_item_id in trainset.all_items():
        if inner_item_id not in rated_items:
            raw_item_id = trainset.to_raw_iid(inner_item_id)

            predicted_rating = selected_model.predict(
                uid=user_id,
                iid=raw_item_id
            ).est

            recommendations.append({
                "item_id": raw_item_id,
                "predicted_rating": predicted_rating
            })

    top_10 = (
        pd.DataFrame(
            recommendations,
            columns=["item_id", "predicted_rating"]
        )
        .sort_values(
            ["predicted_rating", "item_id"],
            ascending=[False, True]
        )
        .head(10)
        .reset_index(drop=True)
    )

    print("Обрана модель:", selected_model_name)
    print("Користувач:", user_id)
    print("Кількість уже оцінених фільмів:", len(rated_items))
    print("Кількість фільмів-кандидатів:", len(recommendations))

    display(top_10)

Обрана модель: SVD
Користувач: 196
Кількість уже оцінених фільмів: 29
Кількість фільмів-кандидатів: 1624


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

У лабораторній роботі побудовано та порівняно рекомендаційні моделі SVD і KNNBasic на наборі MovieLens 100K. За результатами перехресної перевірки обрано SVD, яка отримала менший RMSE — 0,9425 порівняно з 1,0272 для KNNBasic. Найкращими параметрами SVD стали 50 прихованих факторів, швидкість навчання 0,01 і регуляризація 0,05. Для KNNBasic найкращий результат отримано за використання 40 сусідів, мінімальної кількості сусідів 1 та косинусної схожості між користувачами. На незалежній тестовій вибірці SVD отримала RMSE = 0,9192 і MAE = 0,7234, а KNNBasic — RMSE = 1,0120 і MAE = 0,7984. Тестовий RMSE виявився нижчим за результат перехресної перевірки на 0,0233 для SVD і на 0,0152 для KNNBasic. Для користувача 196 сформовано 10 рекомендацій серед 1624 фільмів-кандидатів із виключенням 29 фільмів, уже оцінених ним у навчальній вибірці. Обмеженнями є нестача інформації про нових користувачів і фільми, неврахування зміни вподобань із часом та те, що RMSE і MAE безпосередньо не оцінюють якість ранжування рекомендацій.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [42]:
class UserBasedRecommender:
    def __init__(self, k=40, rating_scale=(1, 5)):
        self.k = k
        self.rating_scale = rating_scale

    def fit(self, frame):
        matrix = frame.pivot(
            index="user_id",
            columns="item_id",
            values="rating"
        )

        self.user_ids = matrix.index.tolist()
        self.item_ids = matrix.columns.tolist()

        self.user_index = {
            uid: index for index, uid in enumerate(self.user_ids)
        }

        self.item_index = {
            iid: index for index, iid in enumerate(self.item_ids)
        }

        self.ratings = matrix.to_numpy(dtype=float)
        self.observed = ~np.isnan(self.ratings)

        self.global_mean = float(frame["rating"].mean())
        self.user_means = np.nanmean(self.ratings, axis=1)
        self.item_means = np.nanmean(self.ratings, axis=0)

        self.centered = np.where(
            self.observed,
            self.ratings - self.user_means[:, None],
            0.0
        )

        norms = np.linalg.norm(self.centered, axis=1)

        normalized = np.divide(
            self.centered,
            norms[:, None],
            out=np.zeros_like(self.centered),
            where=norms[:, None] > 0
        )

        self.similarity = normalized @ normalized.T
        np.fill_diagonal(self.similarity, 0.0)

        return self

    def predict(self, uid, iid):
        user_index = self.user_index.get(uid)
        item_index = self.item_index.get(iid)

        if user_index is None:
            estimate = (
                self.item_means[item_index]
                if item_index is not None
                else self.global_mean
            )

        elif item_index is None:
            estimate = self.user_means[user_index]

        else:
            similarities = self.similarity[user_index]

            neighbors = np.flatnonzero(
                self.observed[:, item_index]
                & (similarities > 0)
            )

            if len(neighbors) == 0:
                estimate = self.user_means[user_index]
            else:
                neighbors = neighbors[
                    np.argsort(similarities[neighbors])[-self.k:]
                ]

                weights = similarities[neighbors]
                deviations = self.centered[neighbors, item_index]

                estimate = (
                    self.user_means[user_index]
                    + np.dot(weights, deviations) / weights.sum()
                )

        return float(np.clip(estimate, *self.rating_scale))

    def recommend(self, uid, n=10):
        if uid not in self.user_index:
            raise ValueError(
                f"Користувача {uid} немає в навчальній вибірці"
            )

        user_index = self.user_index[uid]

        unseen_items = np.flatnonzero(
            ~self.observed[user_index]
        )

        recommendations = [
            {
                "item_id": self.item_ids[item_index],
                "predicted_rating": self.predict(
                    uid,
                    self.item_ids[item_index]
                )
            }
            for item_index in unseen_items
        ]

        return (
            pd.DataFrame(
                recommendations,
                columns=["item_id", "predicted_rating"]
            )
            .sort_values(
                ["predicted_rating", "item_id"],
                ascending=[False, True]
            )
            .head(n)
            .reset_index(drop=True)
        )

In [43]:
custom_model = UserBasedRecommender(
    k=40,
    rating_scale=reader.rating_scale
)

custom_model.fit(train_df)

custom_predictions = np.array([
    custom_model.predict(uid, iid)
    for uid, iid, rating in testset
])

true_ratings = np.array([
    rating for uid, iid, rating in testset
])

custom_rmse = float(
    np.sqrt(np.mean((true_ratings - custom_predictions) ** 2))
)

custom_mae = float(
    np.mean(np.abs(true_ratings - custom_predictions))
)

all_results = pd.concat([
    results,
    pd.DataFrame({
        "Model": ["Власний алгоритм"],
        "RMSE": [custom_rmse],
        "MAE": [custom_mae]
    })
], ignore_index=True)

all_results = (
    all_results
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(all_results)

print(f"Рекомендації власного алгоритму для користувача {user_id}:")

try:
    custom_top_10 = custom_model.recommend(user_id, n=10)
except ValueError as error:
    print(error)
else:
    display(custom_top_10)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,Власний алгоритм,0.933676,0.728167
2,KNNBasic,1.011953,0.798421


Рекомендації власного алгоритму для користувача 196:


,item_id,predicted_rating
0,1103,5.000000
1,1306,5.000000
2,1500,5.000000
3,1554,5.000000
4,1628,5.000000
5,1629,5.000000
6,1642,5.000000
7,1398,4.980296
8,1467,4.957341
9,1653,4.871921


Реалізовано власний рекомендаційний алгоритм на основі схожості користувачів без використання готових рекомендаційних моделей. Для визначення схожості оцінки центровано відносно середньої оцінки кожного користувача та обчислено косинусну схожість отриманих векторів. Прогноз формується на основі оцінок до 40 найбільш схожих користувачів із додатною схожістю, які оцінили відповідний фільм. На незалежній тестовій вибірці отримано RMSE = 0,9337 і MAE = 0,7282, тому власний алгоритм поступається SVD, але перевершує KNNBasic за обома показниками. Для користувача 196 сформовано топ-10 рекомендацій без фільмів, уже оцінених ним у навчальній вибірці. За відсутності відповідних сусідів використовується середня оцінка користувача, а для невідомого користувача — середня оцінка фільму або загальна середня оцінка.